# Rung 1 — Fitting the whole curve

**Question.** What are `beta` and `gamma` separately?

Rung 0 could not answer this, because a straight line through the early phase only sees
their difference. Using the whole curve means solving the ODE inside an optimiser, and it
forces a choice that rung 0 never had to make: *what does "best fit" mean?*

Two answers, and the gap between them is the content of this rung.

| objective | what it assumes | what it does to epidemic counts |
|---|---|---|
| sum of squares | Gaussian noise, constant variance | the peak dominates; the early phase is ignored |
| negative log-likelihood | the observation model actually in use | weights each point by its expected noise |

They coincide exactly when the noise is Gaussian with constant variance. Count data is
neither.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from inferladder import default_sir_dataset, default_sir_model
from inferladder.inference import fit, negative_log_likelihood, sum_of_squares

dataset = default_sir_dataset()
model = default_sir_model()
truth = dataset.truth

start = dataset.theta_true * 1.4  # deliberately off, so the optimiser has work to do
mle = fit(negative_log_likelihood(model, dataset.y, dataset.t), start, model.param_names)
lsq = fit(sum_of_squares(model, dataset.y, dataset.t), start, model.param_names)

In [ ]:
rows = []
for name in model.param_names:
    rows.append((name, truth[name], mle[name], lsq[name]))

print(f"{'param':8} {'truth':>9} {'MLE':>9} {'least sq':>9}")
for name, tr, m_, l_ in rows:
    print(f"{name:8} {tr:9.4f} {m_:9.4f} {l_:9.4f}")
print()
print(
    f"{'R0':8} {truth['beta'] / truth['gamma']:9.4f} "
    f"{mle['beta'] / mle['gamma']:9.4f} {lsq['beta'] / lsq['gamma']:9.4f}"
)

Least squares is visibly further from the truth — and, importantly, it is not merely
noisier. It is biased in a direction you can predict from what it assumes.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(dataset.t, dataset.y, ".", ms=4, color="#9fb0c0", label="data")
ax.plot(
    dataset.t,
    model.expected(dataset.theta_true, dataset.t),
    "-",
    lw=2,
    color="#3c3c3c",
    label="truth",
)
ax.plot(
    dataset.t,
    model.expected(mle.theta, dataset.t),
    "-",
    lw=2,
    color="#33628f",
    label="maximum likelihood",
)
ax.plot(
    dataset.t,
    model.expected(lsq.theta, dataset.t),
    "--",
    lw=2,
    color="#c1502e",
    label="least squares",
)
ax.set(xlabel="day", ylabel="reported cases")
ax.legend()
ax.grid(alpha=0.3)
fig.tight_layout()

Both curves look reasonable through the peak. The disagreement is in the tails, which
carry most of the information about `gamma` and almost none of the visual weight.

## Why least squares goes wrong here

Plot the residuals against the fitted mean. Under the constant-variance assumption this
should be a horizontal band.

In [ ]:
expected = model.expected(mle.theta, dataset.t)
residuals = dataset.y - expected

fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
axes[0].plot(expected, residuals, ".", ms=5, color="#33628f")
axes[0].axhline(0, color="#c1502e", lw=1)
axes[0].set(xlabel="fitted mean", ylabel="residual", title="raw residuals: a fan, not a band")

axes[1].plot(expected, residuals / np.sqrt(np.maximum(expected, 1)), ".", ms=5, color="#33628f")
axes[1].axhline(0, color="#c1502e", lw=1)
axes[1].set(
    xlabel="fitted mean", ylabel="residual / sqrt(mean)", title="scaled by the expected noise"
)
for ax in axes:
    ax.grid(alpha=0.3)
fig.tight_layout()

The spread grows with the mean. Least squares treats a residual of 50 at the peak as
just as surprising as a residual of 50 at day 10 — but at day 10 it is enormous and at the
peak it is nothing. So the peak dominates the fit and the early phase, where the growth
rate lives, is effectively discarded.

## When to stop here

**This rung is enough if you need a single plausible parameter set** — to run a
counterfactual scenario, or to hand a mechanistic model to someone else.

**It is not enough to know whether that parameter set is trustworthy.** Two questions are
still open, and they are different questions: *did the optimiser actually find the
minimum* (rung 2), and *do the data pin these parameters down at all* (rung 3). Nothing
so far distinguishes a sharp optimum from a flat valley.